In [ ]:
"""
TED Tender Intelligence 
========================================
Daily runner: fetches notices published in the last 24 hours,
scores them against the profile, exports to Excel.

USAGE:
    # In Jupyter (one-off):
    exec(open("ted_intelligence.py").read())
    live, intel = fetch()
    export(live, intel)

    # As a script (called by GitHub Actions daily):
    python ted_intelligence.py
"""

import requests, re, time, os
import pandas as pd
from datetime import datetime, timedelta, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

# ═══════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════

# How far back to look. Default = 1 day for the daily runner.
# Set to 90 for a full backfill when running manually.
DAYS_BACK        = int(os.environ.get("DAYS_BACK", "1"))

PAGE_SIZE        = 250
MAX_PAGES        = 999
MIN_SCORE        = 3
AI_MAX_NOTICES   = 50
AI_WORKERS       = 4

TODAY            = datetime.now(timezone.utc)
DEADLINE_CUTOFF  = TODAY - timedelta(hours=24)

SEARCH_URL   = "https://api.ted.europa.eu/v3/notices/search"
CLAUDE_URL   = "https://api.anthropic.com/v1/messages"
CLAUDE_MODEL = "claude-sonnet-4-20250514"

# Anthropic API key — set as env var in GitHub Actions / Streamlit secrets
# Or paste directly here for local use
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

RESPONSE_FIELDS = [
    "publication-number",
    "notice-title",
    "buyer-name",
    "notice-type",
    "BT-13(t)-Part",
]

# ── Broad search terms (cast wide net, score locally) ────────────
BROAD_SEARCH_TERMS = [
    "innovation", "commercialisation", "valorisation",
    "technology transfer", "advisory", "market study",
    "consultancy", "knowledge transfer", "exploitation",
    "research", "startup", "deep tech",
    "horizon europe", "dissemination",
]

# ── Tier 1: DM core language — +7 per hit ────────────────────────
TIER1 = [
    "technology valorisation", "tech valorisation",
    "technology commercialisation", "tech commercialisation",
    "exploitation of results", "exploitation and dissemination",
    "dissemination and exploitation",
    "commercialisation of research", "commercialisation of innovation",
    "knowledge transfer", "technology transfer",
    "research to market", "research commercialisation",
    "ip commercialisation", "ip to market",
    "market uptake", "market adoption",
    "eic accelerator", "eic business acceleration",
    "eic bas", "business acceleration service",
    "horizon europe", "horizon 2020",
    "eurostars", "eureka cluster",
    "eit digital", "eit manufacturing", "eit health",
    "eit urban mobility", "eit food", "eit rawmaterials",
    "diana programme", "diana accelerator",
    "nato innovation", "defence innovation accelerator",
    "venture building", "venture creation", "venture support",
    "startup support", "startup acceleration",
    "deeptech", "deep tech",
    "spin-off", "spin-out", "spinout",
    "investor readiness", "investment readiness",
    "pre-commercial procurement", "innovation partnership",
    "innovation ecosystem", "ecosystem orchestration",
    "consortium commercialisation", "project commercialisation",
    "innovation management support", "innovation support services",
    "scale-up support", "scaleup support",
    "product-market fit", "market validation",
    "dual-use technology", "dual use technology",
]

# ── Tier 2: Contextual terms — +3 per hit ────────────────────────
TIER2 = [
    "commercialisation", "valorisation", "go-to-market",
    "market intelligence", "market study", "market analysis",
    "competitive analysis", "landscape analysis",
    "technology assessment", "feasibility study",
    "business model", "value proposition",
    "stakeholder mapping", "ecosystem mapping",
    "advisory services", "strategic advisory",
    "fundraising support", "funding strategy",
    "regulatory strategy", "regulatory navigation",
    "market entry", "market access",
    "technology roadmap", "innovation strategy",
    "dual use", "dual-use",
    "artificial intelligence", "machine learning",
    "cybersecurity", "digital twin",
    "quantum", "photonics", "semiconductor",
    "advanced materials", "energy storage",
    "autonomous systems", "robotics",
    "smart grid", "critical infrastructure",
    "space technology", "satellite",
]

BUYER_SIGNALS = [
    "innovation", "research", "universit", "institute",
    "agency", "accelerator", "incubator", "eit", "eic",
    "diana", "science", "nwo", "anr", "bpifrance",
    "vinnova", "enterprise ireland", "innovate uk",
    "rvo", "ffg", "ncbr", "enabel", "tekes",
    "business finland", "horizon", "eureka", "interreg",
]

NEGATIVES = [
    "valorisation énergétique", "valorisation des déchets",
    "valorisation des boues", "valorisation des papiers",
    "valorisation des mâchefers", "valorisation des espaces",
    "valorisation des marques", "valorisation du patrimoine immobilier",
    "valorisation des certificats", "valorisation des actifs immobiliers",
    "unité de valorisation", "centre de valorisation",
    "valorisation foncière", "valorisation du patrimoine bâti",
    "commercialisation de locaux", "commercialisation du patrimoine",
    "commercialisation des actifs", "commercialisation immobilière",
    "gestion locative", "mandat de gestion",
    "locaux commerciaux", "habitations à loyer",
    "patrimoine résidentiel",
    "noise control", "baulärm", "car park", "parking lot",
    "building maintenance", "cleaning service", "catering service",
    "waste management", "refuse collection", "sludge",
    "urée", "déchets ménagers", "bacs roulants", "incineration",
    "fire alarm", "cctv installation", "security guard",
    "grounds maintenance", "road construction", "bridge construction",
    "electrical installation work", "plumbing",
]

LIVE_TYPES  = {"cn-standard","cn-social","cn-desg","cn-tran",
               "pin-cfc-standard","pin-cfc-social","pin-only"}
INTEL_TYPES = {"can-standard","can-social","can-desg",
               "can-tran","can-modif"}

DM_PROFILE = """
DevelopMinded is a deep-tech commercialisation and venture support firm.
They are hired to:
- Support technology valorisation / commercialisation of R&D results
- Deliver exploitation and dissemination workstreams in EU-funded projects
- Provide market intelligence and go-to-market strategy for deep tech
- Support investor readiness, fundraising strategy, venture building
- Coach startups through EIC, Horizon Europe, EIT, DIANA programmes
- Map innovation ecosystems, stakeholder engagement, regulatory navigation

Their technology domains: AI/ML, cybersecurity, digital twins, quantum,
semiconductors, photonics, advanced materials, energy storage,
defence/security/space, autonomous systems, robotics, smart grids.

They are NOT relevant for:
- Physical construction, road works, building maintenance
- Waste management, energy plant operation
- Real estate letting or commercialisation of property
- Security guarding, cleaning, catering
- Standard IT procurement (software licences, off-the-shelf hardware)
- Medical supplies, pharmaceuticals
- Generic legal, audit, or financial audit services
- Website development or IT system maintenance
"""

# ═══════════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════════

def flat(v) -> str:
    if not v: return ""
    if isinstance(v, str): return v.strip()
    if isinstance(v, list):
        return " | ".join(p for p in [flat(i) for i in v] if p)
    if isinstance(v, dict):
        for k in ("eng","ENG","fra","FRA","nld","NLD","deu","DEU"):
            if k in v and v[k]: return flat(v[k])
        for val in v.values():
            s = flat(val)
            if s: return s
    return str(v).strip() if v else ""


def parse_deadline(raw):
    if not raw: return None
    if isinstance(raw, list): raw = raw[0] if raw else None
    if isinstance(raw, dict): raw = next(iter(raw.values()), None)
    if not raw or not isinstance(raw, str): return None
    for fmt in ("%Y-%m-%dT%H:%M:%S%z", "%Y-%m-%dT%H:%M:%S.%f%z",
                "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d"):
        try:
            dt = datetime.strptime(raw[:25], fmt)
            if dt.tzinfo is None:
                dt = dt.replace(tzinfo=timezone.utc)
            return dt
        except ValueError:
            continue
    return None


def score_notice(title: str, buyer: str, ntype: str):
    title_low = title.lower()
    text      = f"{title} {buyer}".lower()
    buyer_low = buyer.lower()
    if any(neg in title_low for neg in NEGATIVES):
        return -999, "skip", [], []
    t1 = [t for t in TIER1 if t in text]
    t2 = [t for t in TIER2 if t in text]
    buyer_boost = 2 if any(sig in buyer_low for sig in BUYER_SIGNALS) else 0
    sc = 7 * len(t1) + 3 * len(t2) + buyer_boost
    if ntype in INTEL_TYPES: sc -= 5
    if sc < MIN_SCORE: return sc, "skip", t1, t2
    if ntype in INTEL_TYPES:           bucket = "Market intelligence"
    elif t1:                           bucket = "Live opportunity"
    elif len(t2) >= 2 and buyer_boost: bucket = "Live opportunity"
    elif len(t2) >= 3:                 bucket = "Possible opportunity"
    else:                              bucket = "skip"
    return sc, bucket, t1, t2


def extract(raw: dict) -> dict:
    pub   = flat(raw.get("publication-number")) or "—"
    title = flat(raw.get("notice-title"))       or "—"
    buyer = flat(raw.get("buyer-name"))         or "—"
    ntype = flat(raw.get("notice-type"))        or "—"
    dl_dt = parse_deadline(raw.get("BT-13(t)-Part"))
    return {
        "pub_num":     pub,
        "title":       title,
        "buyer":       buyer,
        "notice_type": ntype,
        "deadline_dt": dl_dt,
        "deadline":    dl_dt.strftime("%Y-%m-%d") if dl_dt else "—",
        "link":        f"https://ted.europa.eu/en/notice/-/detail/{pub}" if pub != "—" else "",
    }

# ═══════════════════════════════════════════════════════════════
# STEP 1 — FETCH
# ═══════════════════════════════════════════════════════════════

def make_query():
    parts = " OR ".join(f'FT ~ "{k}"' for k in BROAD_SEARCH_TERMS[:14])
    since = (datetime.now() - timedelta(days=DAYS_BACK)).strftime("%Y%m%d")
    return f"({parts}) AND publication-date >= {since}"


def _fetch_page(payload) -> tuple:
    for attempt in range(3):
        try:
            r = requests.post(SEARCH_URL, json=payload, timeout=60)
        except requests.RequestException as e:
            return None, str(e), None, "?"
        if r.status_code == 200:
            d = r.json()
            return (d.get("notices", []), None,
                    d.get("iterationNextToken"),
                    d.get("totalNoticeCount", "?"))
        elif r.status_code == 429:
            wait = 35 * (attempt + 1)
            print(f"\n  Rate limited — waiting {wait}s...")
            time.sleep(wait)
        else:
            return None, f"{r.status_code}: {r.text[:100]}", None, "?"
    return None, "Max retries", None, "?"


def fetch(days_back: int = DAYS_BACK) -> tuple[pd.DataFrame, pd.DataFrame]:
    print("=" * 64)
    print(f"  TED Intelligence — DevelopMinded")
    print(f"  {datetime.now():%Y-%m-%d %H:%M UTC}")
    print(f"  Window: last {days_back} day(s)")
    print("=" * 64)

    # Override DAYS_BACK if passed explicitly
    since = (datetime.now() - timedelta(days=days_back)).strftime("%Y%m%d")
    parts = " OR ".join(f'FT ~ "{k}"' for k in BROAD_SEARCH_TERMS[:14])
    query = f"({parts}) AND publication-date >= {since}"
    print(f"\nQuery:\n{query}\n")
    print("Fetching all pages...\n")

    all_notices, token, page, t0 = [], None, 0, time.time()
    while page < MAX_PAGES:
        payload = {
            "query": query, "fields": RESPONSE_FIELDS,
            "limit": PAGE_SIZE, "scope": "ACTIVE",
            "checkQuerySyntax": False,
            "paginationMode": "ITERATION",
            "onlyLatestVersions": True,
        }
        if token: payload["iterationNextToken"] = token
        notices, err, token, total = _fetch_page(payload)
        if err: print(f"\n  Error: {err}"); break
        if not notices: break
        all_notices.extend(notices)
        page += 1
        print(f"  Page {page:3d} | +{len(notices):3d} | {len(all_notices):,} / {total:,}", end="\r")
        if not token: print(f"  Page {page:3d} | done ✓" + " " * 30); break
        if page % 10 == 0: time.sleep(0.5)

    print(f"\n\nFetched {len(all_notices):,} notices in {time.time()-t0:.0f}s\n")
    if not all_notices: return pd.DataFrame(), pd.DataFrame()

    live_rows, intel_rows = [], []
    n_expired = n_no_dl = n_future = n_neg = n_low = 0

    for raw in all_notices:
        e  = extract(raw)
        dt = e["deadline_dt"]
        if dt is None:             n_no_dl  += 1
        elif dt < DEADLINE_CUTOFF: n_expired += 1; continue
        else:                      n_future += 1
        sc, bucket, t1, t2 = score_notice(e["title"], e["buyer"], e["notice_type"])
        if bucket == "skip":
            if sc == -999: n_neg += 1
            else:          n_low += 1
            continue
        row = {k: v for k, v in e.items() if k != "deadline_dt"}
        row.update(score=sc, bucket=bucket,
                   t1_hits=", ".join(t1), t2_hits=", ".join(t2[:4]))
        (live_rows if bucket != "Market intelligence" else intel_rows).append(row)

    def to_df(rows):
        if not rows: return pd.DataFrame()
        return pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)

    live, intel = to_df(live_rows), to_df(intel_rows)

    print("=" * 64)
    print(f"Deadline : {n_future:,} future | {n_no_dl:,} none | {n_expired:,} expired (dropped)")
    print(f"Scoring  : {n_neg:,} negative | {n_low:,} below threshold")
    print(f"Results  : 🟢 {len(live):,} live | 📊 {len(intel):,} market intel\n")

    if not live.empty:
        print("── LIVE OPPORTUNITIES ──")
        print(live[["score","bucket","deadline","title","buyer","t1_hits"]].to_string(index=False))
    if not intel.empty:
        print("\n── MARKET INTELLIGENCE ──")
        print(intel[["score","deadline","title","buyer","t1_hits"]].to_string(index=False))

    return live, intel


# ═══════════════════════════════════════════════════════════════
# STEP 3 — EXPORT
# ═══════════════════════════════════════════════════════════════

def export(live: pd.DataFrame, intel: pd.DataFrame,
           filename: str = "ted_tenders.xlsx"):
    if live.empty and intel.empty: print("Nothing to export."); return
    live_cols  = ["score","bucket","deadline","title","buyer",
                  "notice_type","t1_hits","ai_reason","link"]
    intel_cols = ["score","deadline","title","buyer",
                  "notice_type","t1_hits","link"]
    with pd.ExcelWriter(filename, engine="openpyxl") as w:
        for df, sheet, cols in [
            (live,  "Live Opportunities",  live_cols),
            (intel, "Market Intelligence", intel_cols),
        ]:
            if df.empty: continue
            out = df[[c for c in cols if c in df.columns]]
            out.to_excel(w, index=False, sheet_name=sheet)
            ws = w.sheets[sheet]
            for col in ws.columns:
                ws.column_dimensions[col[0].column_letter].width = min(
                    max(len(str(c.value or "")) for c in col)+4, 60)
    print(f"Exported → {filename}")
    print(f"  Live Opportunities  : {len(live)}")
    print(f"  Market Intelligence : {len(intel)}")
    return filename

# ═══════════════════════════════════════════════════════════════
# ENTRY POINT — called by GitHub Actions
# ═══════════════════════════════════════════════════════════════

if __name__ == "__main__":
    live, intel = fetch()
    # Uncomment to enable AI filter (needs ANTHROPIC_API_KEY env var):
    # live = ai_filter(live)
    export(live, intel)